In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os

In [ ]:
cale_date = "../data/raw/Crop_recommendation.csv"
df = pd.read_csv(cale_date)

X = df.drop('label', axis=1)
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import pandas as pd

modele = {
    "KNN": KNeighborsClassifier(),
    "SVM (RBF Kernel)": SVC(kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost (Ales)": xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', max_depth=5, learning_rate=0.1, random_state=42, n_jobs=1)
}

comparatie_rezultate = []

for nume, mod in modele.items():
    mod.fit(X_train_scaled, y_train)
    sc_train = mod.score(X_train_scaled, y_train)
    sc_test = mod.score(X_test_scaled, y_test)
    comparatie_rezultate.append({
        "Model": nume,
        "Acuratețe Train": f"{sc_train * 100:.2f}%",
        "Acuratețe Test": f"{sc_test * 100:.2f}%"
    })

df_comparatie = pd.DataFrame(comparatie_rezultate)
print("=== REZULTATE COMPARATIVE CONCURS ===")
print(df_comparatie.to_string(index=False))

In [ ]:
model_xgb = modele["XGBoost (Ales)"]
xgb_acc = model_xgb.score(X_test_scaled, y_test)
print(f"Modelul selectat pentru producție: XGBoost cu acuratețea de {xgb_acc * 100:.2f}%")

In [ ]:
y_pred = model_xgb.predict(X_test_scaled)
f1_ponderat = f1_score(y_test, y_pred, average='weighted')
print(f1_ponderat)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Greens',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Matricea de Confuzie - Evaluare Model Tabular (XGBoost)')
plt.ylabel('Cultură Reală')
plt.xlabel('Cultură Predictată')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

os.makedirs('../models', exist_ok=True)
joblib.dump(model_xgb, '../models/xgboost_soil_model.pkl')
joblib.dump(scaler, '../models/soil_scaler.pkl')
joblib.dump(label_encoder, '../models/soil_label_encoder.pkl')